In [1]:
import os
from pathlib import Path

BASE_DIR = Path.cwd()
ROOT_DIR = BASE_DIR.parent
CACHE_DIR = str(ROOT_DIR.parent / "huggingface_cache")

os.environ["HF_HOME"] = CACHE_DIR
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import sys
import random
import json
from tqdm import tqdm
import torch
from dotenv import load_dotenv
from transformers import set_seed
from unsloth import FastLanguageModel, get_chat_template
load_dotenv()

SEED = 42
set_seed(SEED)
random.seed(SEED)

BATCH_SIZE = 8

sys.path.insert(0, str(ROOT_DIR))

DATA_MCQ_DIR = (ROOT_DIR / "data/eval_data/tele-qna.json").resolve()
DATA_OQNA_DIR = (ROOT_DIR / "data/eval_data/tele-eval-10k.json").resolve()
DATA_MCQ_CUS_DIR = (ROOT_DIR / "data/eval_data/tele-qna-custom.json").resolve()
RESULT_MCQ_DIR = (ROOT_DIR / "data/benchmark_results/tele-qna-with-qwen3-original.json").resolve()
RESULT_OQNA_DIR = (ROOT_DIR / "data/benchmark_results/tele-eval-with-qwen3-original.json").resolve()
RESULT_MCQ_CUS_DIR = (ROOT_DIR / "data/benchmark_results/tele-qna-cus-with-qwen3-original.json").resolve()
MODEL_DIR = (ROOT_DIR / "models").resolve()

/tmp/ipykernel_16161/2979454590.py:18: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel, get_chat_template


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [5]:
import unsloth
from unsloth import FastLanguageModel, get_chat_template

def get_model_and_tokenizer():
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="unsloth/Qwen3-8B",
        max_seq_length=2048,
        load_in_4bit=False,
    )
    tokenizer = get_chat_template(tokenizer, chat_template="qwen-3")
    return model, tokenizer

In [6]:
model, tokenizer = get_model_and_tokenizer()

==((====))==  Unsloth 2026.4.2: Fast Qwen3 patching. Transformers: 4.57.6. vLLM: 0.19.0.
   \\   /|    NVIDIA L40. Num GPUs = 2. Max memory: 44.392 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.58G [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

unsloth/Qwen3-8B does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [5]:
import json
import re
from datasets import Dataset
from tqdm import tqdm

with open(DATA_MCQ_CUS_DIR, "r", encoding="utf-8") as f:
    data = json.load(f)

dataset = Dataset.from_list(data)

In [6]:
def build_texts_mcq(batch):
    texts = []

    for sample in batch:
        choices_text = "\n".join(
            [f"{i+1}. {c}" for i, c in enumerate(sample["choices"])]
        )

        messages = [
            {
                "role": "user",
                "content": f"""You are a multiple choice QA system.

Select exactly ONE correct answer.

Output format (STRICT):
Đáp án là X

Question:
{sample['question']}

Choices:
{choices_text}
"""
            }
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )

        texts.append(text)

    return texts

In [7]:
def generate_batch_mcq(texts):
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False,
        temperature = 0.7,
        top_p = 0.8, 
        top_k = 20,
    )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    return decoded

In [9]:
import json
from tqdm import tqdm

with open(RESULT_MCQ_CUS_DIR, "w", encoding="utf-8") as fout:
    fout.write("[\n")

    total = len(dataset)

    for start in tqdm(range(0, total, BATCH_SIZE)):
        batch = dataset.select(range(start, min(start + BATCH_SIZE, total)))
        batch = list(batch)

        texts = build_texts_mcq(batch)
        decoded = generate_batch_mcq(texts)

        for i, sample in enumerate(batch):
            full_text = decoded[i]

            result = {
                "question": sample["question"],
                "choices": sample["choices"],
                "answer": sample["answer"],
                "category": sample.get("category"),
                "explaination": sample.get("explaination"),
                "model_output": full_text,
            }

            json.dump(result, fout, ensure_ascii=False)

            global_idx = start + i
            if global_idx < total - 1:
                fout.write(",\n")
            else:
                fout.write("\n")

        fout.flush()

    fout.write("]")

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 250/250 [05:40<00:00,  1.36s/it]


In [13]:
import json
import re
from collections import defaultdict

def extract_choice(text):
    if not text:
        return None
    match = re.search(r"\b([1-4])\b", text)
    return int(match.group(1)) if match else None

with open((RESULT_MCQ_CUS_DIR).resolve(), "r", encoding="utf-8") as f:
    data = json.load(f)



total = 0
correct = 0

cat_stats = defaultdict(lambda: {"total": 0, "correct": 0})

def clean_output(text):
    if not text:
        return ""

    marker = "\n\nassistant\n<think>\n\n</think>\n\n"
    if marker in text:
        text = text.split(marker)[1]

    return text.strip()


for idx, sample in enumerate(data):
    gt = sample.get("answer")

    raw_text = sample.get("model_output") or sample.get("think", "")
    text = clean_output(raw_text)

    pred = extract_choice(text)

    total += 1
    cat = sample.get("category") or "UNKNOWN"

    cat_stats[cat]["total"] += 1

    if pred == gt:
        correct += 1
        cat_stats[cat]["correct"] += 1


print("=== OVERALL ===")
if total > 0:
    print(f"Accuracy: {correct}/{total} = {correct/total:.4f}")
else:
    print("No data")

print("\n=== BY CATEGORY ===")
for cat, stat in cat_stats.items():
    t = stat["total"]
    c = stat["correct"]
    acc = c / t if t > 0 else 0
    print(f"{cat}: {c}/{t} = {acc:.4f}")

=== OVERALL ===
Accuracy: 1731/2000 = 0.8655

=== BY CATEGORY ===
Standards Specifications: 1267/1477 = 0.8578
Standards Overview: 214/241 = 0.8880
Lexicon: 196/226 = 0.8673
Research Overview: 21/22 = 0.9545
Research Publications: 33/34 = 0.9706


In [6]:
import json
import re
from datasets import Dataset
from tqdm import tqdm

with open(DATA_OQNA_DIR, "r", encoding="utf-8") as f:
    data = json.load(f)

dataset = Dataset.from_list(data)

In [8]:
def build_texts_qna(batch):
    texts = []

    for sample in batch:
        messages = [
            {
                "role": "user",
                "content": f"""Answer this question

Question:
{sample['question']}

Choices:
{sample['answer']}
"""
            }
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )

        texts.append(text)

    return texts

In [9]:
def generate_batch_qna(texts):
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=1024,
        do_sample=False,
        temperature = 0.7,
        top_p = 0.8, 
        top_k = 20,
    )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    return decoded

In [ ]:
import json
from tqdm import tqdm

with open(RESULT_OQNA_DIR, "w", encoding="utf-8") as fout:
    fout.write("[\n")

    total = len(dataset)

    for start in tqdm(range(0, total, BATCH_SIZE)):
        batch = dataset.select(range(start, min(start + BATCH_SIZE, total)))
        batch = list(batch)

        texts = build_texts_qna(batch)
        decoded = generate_batch_qna(texts)


        MARKER = "\n\nassistant\n<think>\n\n</think>\n\n"

        for i, sample in enumerate(batch):
            full_text = decoded[i]
        
            if MARKER in full_text:
                full_text = full_text.split(MARKER)[0]
        
            result = {
                "question": sample["question"],
                "answer": sample["answer"],
                "type": sample.get("type"),
                "model_output": full_text,
            }

        json.dump(result, fout, ensure_ascii=False)

        global_idx = start + i
        if global_idx < total - 1:
            fout.write(",\n")
        else:
            fout.write("\n")

        fout.flush()

    fout.write("]")

In [ ]:
import json
from collections import defaultdict
from rouge_score import rouge_scorer
import pandas as pd

def evaluate_rouge_by_type(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    scorer = rouge_scorer.RougeScorer(
        ["rouge1", "rouge2", "rougeL"],
        use_stemmer=True
    )

    scores = defaultdict(lambda: {"r1": [], "r2": [], "rl": []})

    for item in data:
        pred = item.get("model_output", "")
        ref = item.get("answer", "")
        t = item.get("type", "unknown")

        if not pred or not ref:
            continue

        s = scorer.score(ref, pred)

        scores[t]["r1"].append(s["rouge1"].fmeasure)
        scores[t]["r2"].append(s["rouge2"].fmeasure)
        scores[t]["rl"].append(s["rougeL"].fmeasure)

    rows = []
    for t, v in scores.items():
        rows.append({
            "type": t,
            "count": len(v["r1"]),
            "rouge1": sum(v["r1"]) / len(v["r1"]),
            "rouge2": sum(v["r2"]) / len(v["r2"]),
            "rougeL": sum(v["rl"]) / len(v["rl"]),
        })

    df = pd.DataFrame(rows).sort_values("type")
    return df

In [ ]:
df = evaluate_rouge_by_type(RESULT_OQNA_OR_DIR)
df